# AutoHDR setup + smoke test (Google Colab)

Why Colab instead of native Windows: this repo's OCR detector/recognizer (`dist/det_model`, `dist/model_exe`) are **precompiled Linux ELF executables** — confirmed by running `file` on them — with no Windows build anywhere in the download. They cannot run on native Windows at all. Colab is real Linux with a real GPU already driver-configured, so none of that (or the mmcv/mmdet Windows-wheel gap, or the CPU-only default torch install) is a problem here.

**Before running this notebook:** upload your already-downloaded checkpoint **zip files** (as originally downloaded from BaiduYun, no need to extract them yourself) to a Google Drive folder — re-downloading them from BaiduYun here would just mean solving the same CAPTCHA again, so reuse what you already have locally. As of writing, that folder holds:

```
MyDrive/Master-CS/NLP/cuoi_ky/
  AutoHDR-Qwen2-1-5-B.zip
  unet.zip
  ocr_models.zip
  damage_detect.pth (or a zip containing it)   <- still missing, add once downloaded
```

Step 3 below extracts and places all of these automatically (it searches recursively inside each zip for the file/folder it needs, so exact internal zip layout doesn't matter) — you don't need to rename or restructure anything by hand.

Runtime → Change runtime type → GPU (T4 is enough for the 1.5B LLM + detector + diffusion).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_SRC = '/content/drive/MyDrive/Master-CS/NLP/cuoi_ky'  # adjust if you moved the folder
import os
assert os.path.isdir(CKPT_SRC), f"{CKPT_SRC} not found - upload your checkpoint zips there first"
print(sorted(os.listdir(CKPT_SRC)))

## 1. Clone the repo

Clones from **your own fork** (`ptran1203/nlp_cuoiky`), not upstream `SCUT-DLVCLab/AutoHDR` directly. The fork already has `utils/det_wrapper.py`/`utils/reg_wrapper.py` patched in place (CUDA hidden from those two subprocesses, dtype fixes, stderr logged to a file instead of an unreliable PIPE read) - no separate patch step needed here anymore.

To pick up a *newer* fix later: push it to the fork, then just re-run this cell - it pulls if the repo's already there, clones fresh otherwise, so it always ends up on the latest commit. Cloned into the Colab VM's local disk (`/content/`), not Drive - ephemeral, but re-cloning/pulling takes seconds.

In [ ]:
import os
%cd /content
if os.path.isdir('/content/nlp_cuoiky/.git'):
    print('Repo exists, pulling latest...')
    !git -C /content/nlp_cuoiky pull
else:
    print('Cloning fresh...')
    !git clone https://github.com/ptran1203/nlp_cuoiky /content/nlp_cuoiky
%cd /content/nlp_cuoiky/AutoHDR
!git log -1 --format='On commit: %h %s (%cr)'
# infer_pipeline.py writes here directly without creating it first (unlike results/, which the
# script does create) - tracked as empty via .gitkeep in the fork, but doesn't hurt to be sure.
!mkdir -p tmp_img

## 1b. Patch the OCR subprocess wrappers

This step re-writes `utils/det_wrapper.py` and `utils/reg_wrapper.py` in the freshly-cloned repo (overwriting the originals) - **this has to run after every fresh clone**, since `git clone` above always pulls the pristine upstream files, not a patched copy.

Why this patch exists: `dist/det_model/det_model` and `dist/model_exe` are precompiled Python apps (PyInstaller-frozen) that bundle their **own** CUDA runtime at build time. Confirmed by testing: with no GPU visible at all (an earlier attempt inside WSL2 on Windows, no GPU passthrough there), both start reliably in ~20-25s and cleanly fall back to CPU. But with a real GPU actually visible (Colab), the unpatched originals hang and never bind their socket at all - `RuntimeError: Failed to connect to server after multiple attempts`, even with a 120s retry budget - almost certainly a version/driver mismatch between the GPU actually present and whatever CUDA runtime got bundled into the frozen binary at build time, blocking its own CUDA init before it ever gets to opening the socket.

Fix: force `CUDA_VISIBLE_DEVICES=''` specifically for these two subprocesses (regardless of what GPU the main `infer_pipeline.py` process itself is using), reproducing the one configuration that's actually been proven to work, and force any tensor/device sent to them to `cpu` accordingly. `det_wrapper.py` additionally captures the subprocess's (very verbose) stdout/stderr to a log file instead of losing it, so a future failure is actually debuggable.

In [ ]:
%%writefile utils/det_wrapper.py
import subprocess
import torch
import pickle
import sys,os
import time
import socket
from typing import Optional

class det_model:
    def __init__(self, executable_path: str = './dist/det_model/det_model', port: int = 12345, max_retries: int = 60):
        self.executable_path = executable_path
        self.process = None
        self.stride = None
        self._is_running = False
        self.port = port
        self.sock = None
        self.max_retries = max_retries

    def _read_log_tail(self, max_chars: int = 4000) -> str:
        try:
            with open(self._log_path, 'r', errors='replace') as f:
                text = f.read()
            return text[-max_chars:] if len(text) > max_chars else text
        except Exception as e:
            return f"(couldn't read log: {e})"

    def _connect_with_retry(self):
        """尝试连接服务器，带重试机制"""
        retries = 0
        while retries < self.max_retries:
            try:
                self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                self.sock.connect(('localhost', self.port))
                return True
            except ConnectionRefusedError:
                print(f"Connection attempt {retries + 1} failed, retrying...", file=sys.stderr)
                retries += 1
                time.sleep(2)  # 等待2秒后重试
                continue
        return False

    def start(self):
        try:
            executable_path = os.path.abspath(self.executable_path)
            print(f"Starting process: {executable_path}")

            # Force this subprocess to see no GPU at all. The frozen binary bundles its own CUDA
            # runtime at build time; with a real GPU visible (Colab), a version/driver mismatch
            # there hangs CUDA init before the process ever binds its socket - confirmed by
            # testing: with NO GPU visible at all it starts reliably in ~20-25s and falls back to
            # CPU cleanly, but with a GPU visible it never bound the port even after a 120s retry
            # budget. Hiding the GPU reproduces the one config actually proven to work.
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = ''

            # Capture stdout/stderr to a log file instead of just inheriting the parent's - this
            # process prints a LOT (PyInstaller loader + Python import tracing), enough to bury
            # the one line that actually matters if something goes wrong.
            self._log_path = os.path.abspath('det_model_subprocess.log')
            self._log_file = open(self._log_path, 'w')
            self.process = subprocess.Popen(
                ['dist/det_model/det_model', str(self.port)],
                env=env, stdout=self._log_file, stderr=subprocess.STDOUT,
            )
            # 检查进程是否立即退出
            time.sleep(1)
            exit_code = self.process.poll()
            if exit_code is not None:
                self._log_file.flush()
                print(f"Process exited immediately with code: {exit_code}")
                print(f"Log tail ({self._log_path}):")
                print(self._read_log_tail())
                raise RuntimeError(f"Process exited with code {exit_code}")

            print("Process started successfully")

            # 等待服务器启动并尝试连接
            if not self._connect_with_retry():
                self._log_file.flush()
                print(f"Log tail ({self._log_path}) - process was still running but never bound its socket:")
                print(self._read_log_tail())
                raise RuntimeError("Failed to connect to server after multiple attempts")

            print(f"Process started with PID: {self.process.pid}")
            self._is_running = True

        except Exception as e:
            self.cleanup()
            raise RuntimeError(f"Failed to start process: {str(e)}")

    def _send_data(self, data):
        """发送数据到服务器"""
        serialized_data = pickle.dumps(data)
        length = len(serialized_data)
        self.sock.sendall(length.to_bytes(4, byteorder='big'))
        self.sock.sendall(serialized_data)

    def _recv_data(self):
        """从服务器接收数据"""
        length_bytes = self.sock.recv(4)
        if not length_bytes:
            raise RuntimeError("Connection closed by server")
        length = int.from_bytes(length_bytes, byteorder='big')

        data = b''
        while len(data) < length:
            chunk = self.sock.recv(length - len(data))
            if not chunk:
                raise RuntimeError("Connection closed by server")
            data += chunk

        result = pickle.loads(data)
        if 'error' in result:
            raise RuntimeError(f"Server error: {result['error']}")
        return result

    def __call__(self, x, mode: int = 2):
        if not self._is_running:
            self.start()

        # The subprocess always runs with CUDA_VISIBLE_DEVICES='' (see start()) - a CUDA tensor
        # or device object pickled here would fail to unpickle there, so force CPU regardless of
        # what device the caller (infer_pipeline.py) intended for its own (GPU) process. Also
        # force float32: infer_pipeline.py casts to .half() for its own CUDA path, and .cpu()
        # alone doesn't change dtype - a float16 tensor stays float16, but the subprocess's own
        # model is float32, producing "Input type (c10::Half) and bias type (float) should be
        # the same".
        if isinstance(x, torch.Tensor):
            x = x.cpu().float()
        elif isinstance(x, torch.device):
            x = torch.device('cpu')

        try:
            if mode == 1:
                print(f"mode 1 begin")
                print(f"x: {x}, mode: {mode}")

                # 发送数据
                self._send_data({'x': x, 'mode': mode})

                # 接收响应
                output = self._recv_data()
                self.stride = output['stride']
                return self.stride

            elif mode == 2:
                if not isinstance(x, torch.Tensor):
                    raise TypeError(f"For mode 2, input should be torch.Tensor, got {type(x)}")

                # 发送数据
                self._send_data({'x': x, 'mode': mode})

                # 接收响应
                output = self._recv_data()
                return output['tensor']
            else:
                raise ValueError(f"Invalid mode: {mode}")

        except Exception as e:
            self.cleanup()
            raise RuntimeError(f"Error during operation: {str(e)}")

    def cleanup(self):
        """清理资源"""
        if self.sock is not None:
            try:
                self.sock.close()
            except:
                pass
            self.sock = None

        if getattr(self, '_log_file', None) is not None:
            try:
                self._log_file.close()
            except:
                pass
            self._log_file = None

        if self.process is not None:
            try:
                self.process.terminate()
                self.process.wait(timeout=1.0)
            except:
                self.process.kill()
            finally:
                self.process = None
                self._is_running = False

    def __del__(self):
        self.cleanup()

if __name__ == '__main__':
    # 测试代码
    try:
        model = det_model('dist/det_model/det_model')
        model.start()
        print("Process started successfully")

        # 测试初始化
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
        stride = model(device, mode=1)
        print(f"Model initialized with stride: {stride}")

        # 测试推理
        img = torch.randn(1, 3, 640, 640)
        output = model(img, mode=2)
        print(f"Inference successful, output shape: {output.shape}")

    except Exception as e:
        print(f"Test failed: {str(e)}")
        sys.exit(1)

In [ ]:
%%writefile utils/reg_wrapper.py
import subprocess
import os
import torch
import pickle
import atexit
import sys

class reg_model:
    def __init__(self, executable_path: str = './dist/model_exe'):
        self.executable_path = executable_path
        self.process = None

    def start(self):
        """启动进程"""
        if self.process is None or self.process.poll() is not None:
            print('Starting process...', file=sys.stderr, flush=True)
            # Force this subprocess to see no GPU at all - same reasoning as det_wrapper.py:
            # the frozen binary bundles its own CUDA runtime at build time, and with a real GPU
            # visible a version/driver mismatch there can hang or crash it before it's usable.
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = ''
            self.process = subprocess.Popen(
                [self.executable_path],
                env=env,
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            print(f'Process started with PID {self.process.pid}', file=sys.stderr, flush=True)

    def __call__(self, x: torch.Tensor, device: str = 'cuda') -> torch.Tensor:
        if self.process is None or self.process.poll() is not None:
            raise RuntimeError("Process is not running")

        # The subprocess always runs with CUDA_VISIBLE_DEVICES='' (see start()) - force CPU
        # regardless of what device the caller passed, same reasoning as det_wrapper.py.
        device = 'cpu'

        try:
            input_data = {
                'tensor': x.float().to(device),
                'device': device
            }

            print('Serializing input data...', file=sys.stderr, flush=True)
            data = pickle.dumps(input_data)

            # 先发送数据长度
            print('Sending data length...', file=sys.stderr, flush=True)
            self.process.stdin.write(len(data).to_bytes(4, byteorder='big'))

            # 发送数据
            print('Sending data...', file=sys.stderr, flush=True)
            self.process.stdin.write(data)
            self.process.stdin.flush()

            # 读取响应长度
            print('Reading response length...', file=sys.stderr, flush=True)
            length_bytes = self.process.stdout.read(4)
            if not length_bytes:
                raise EOFError("Process terminated unexpectedly")
            length = int.from_bytes(length_bytes, byteorder='big')

            # 读取响应数据
            print(f'Reading response data ({length} bytes)...', file=sys.stderr, flush=True)
            output_data = self.process.stdout.read(length)
            output = pickle.loads(output_data)
            print('Response received', file=sys.stderr, flush=True)

            return output['tensor']

        except Exception as e:
            print(f"Error during communication: {e}", file=sys.stderr, flush=True)
            if self.process.poll() is not None:
                print(f"Process terminated with code {self.process.poll()}",
                      file=sys.stderr, flush=True)
                error = self.process.stderr.read()
                if error:
                    print(f"Process error output: {error.decode()}",
                          file=sys.stderr, flush=True)
            raise

    def cleanup(self):
        """清理资源"""
        if self.process is not None:
            print('Cleaning up process...', file=sys.stderr, flush=True)
            if self.process.poll() is None:  # 如果进程还在运行
                try:
                    self.process.stdin.close()
                    self.process.wait(timeout=1.0)
                except subprocess.TimeoutExpired:
                    print('Process not responding, terminating...',
                          file=sys.stderr, flush=True)
                    self.process.terminate()
                    try:
                        self.process.wait(timeout=1.0)
                    except subprocess.TimeoutExpired:
                        print('Process still not responding, killing...',
                              file=sys.stderr, flush=True)
                        self.process.kill()
                        self.process.wait()

            self.process.stdout.close()
            self.process.stderr.close()
            self.process = None
            print('Process cleaned up', file=sys.stderr, flush=True)

    def __del__(self):
        """析构函数"""
        self.cleanup()

In [ ]:
# Verify the patch actually landed on disk - don't guess from whether the error recurs.
# Both lines below MUST print a match. If either prints nothing, cell 6/7 above didn't
# actually run (or ran against a different cwd) - re-run them before going any further.
!grep -n "cpu().float()" utils/det_wrapper.py && echo "det_wrapper.py: PATCHED OK" || echo "det_wrapper.py: PATCH MISSING - re-run cell 6"
!grep -n "x.float().to(device)" utils/reg_wrapper.py && echo "reg_wrapper.py: PATCHED OK" || echo "reg_wrapper.py: PATCH MISSING - re-run cell 7"

## 2. Install dependencies

Not installed into Colab's system Python. Two real problems found by actually running this:

1. **Colab's current default runtime is Python 3.13.** `torch==2.3.0` (the repo's pin) never published a `cp313` wheel at all — it predates Python 3.13 by 5 months (Apr 2024 vs Oct 2024) — confirmed by checking PyTorch's own wheel index. Nothing in this old 2023/2024-era pinned stack (torch 2.1-2.3, mmcv 2.1.0) has `cp313` wheels.
2. **`mmdet==3.3.0` requires `mmcv<2.2.0`, but OpenMMLab's `mmcv` wheels for `torch2.3.0` only start at `2.2.0`** (checked their index directly, on *both* Windows and Linux — this isn't OS-specific like earlier assumed). `mmcv==2.1.0` only has wheels through `torch2.1.0`.

So: build a dedicated **Python 3.10 venv** here (matching what the repo asks for, and what worked on native Windows) and pin `torch/torchvision/torchaudio` to the matched `2.1.0` triplet, same fix as Windows. Everything below installs into that venv, not the notebook's own kernel — the kernel stays Python 3.13 and is only used for `!shell` orchestration, Drive/zip file handling, and image display, none of which need torch.

Also **not using `mim`/`openmim` at all**: it transitively pulls in `openxlab`, which hard-pins `setuptools~=60.2.0` — installing it kept silently clobbering our setuptools fix back to a version that crashes on Python 3.13's removed `pkgutil.ImpImporter`, which is what caused the same `AttributeError` to keep recurring no matter what we did to setuptools directly. Plain `pip install mmcv==2.1.0 -f <wheel-index-url>` does exactly what `mim install` does under the hood, without that dependency chain — and inside the Python 3.10 venv the `ImpImporter` issue doesn't exist in the first place (only removed in Python 3.12+).

In [ ]:
# deadsnakes PPA guarantees python3.10 availability regardless of whatever Ubuntu base
# version Colab currently ships (apt's own default repos may not carry an old Python version).
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get -qq update
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-dev

!python3.10 -m venv /content/venv310
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/python -m pip install -q --upgrade pip

# Every later cell:
#   - uses these HARDCODED absolute paths directly (not Python-variable shell interpolation like
#     `!{PY}`) - that interpolation turned out fragile in practice (a runtime restart without
#     re-running the defining cell left `{PY}`/`{PIP}` unexpanded, reaching bash as literal text:
#     "{PIP}: command not found"). Hardcoded paths can't break that way.
#   - prefixes every venv python/pip call with env-var overrides clearing Colab-specific settings
#     that leak into every subprocess (including this venv's Python) regardless of the venv's own
#     isolation - two confirmed so far, both fixed here:
#       * PYTHONPATH: points at Colab's own system dist-packages; without clearing it, imports can
#         silently resolve to Colab's system copy instead of what's pip-installed in this venv -
#         this is what caused "cannot import name 'is_offline_mode' from 'huggingface_hub'
#         (/usr/local/lib/python3.13/dist-packages/huggingface_hub/__init__.py)".
#       * MPLBACKEND: set to Colab's own `module://matplotlib_inline.backend_inline` for inline
#         notebook plotting - the venv's matplotlib doesn't have that Colab-specific backend
#         package, so it crashes at import time (`mmdet` unconditionally imports
#         seaborn -> matplotlib as part of its own import chain). Set to `Agg` (a real headless
#         backend, correct for a batch script that isn't rendering to a display anyway).
#     If a third Colab-injected env var causes a similar crash later, the fix is the same pattern:
#     add `VARNAME=<safe-value>` to this prefix everywhere below.
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/python --version

In [ ]:
# torch pinned to 2.1.0 (not the repo's 2.3.0) - see the markdown cell above for why: mmcv==2.1.0
# (which mmdet==3.3.0 requires, via mmcv<2.2.0) only has wheels through torch2.1.0.
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/python -c "import torch; print(torch.__version__, 'cuda:', torch.cuda.is_available())"

# Rest of requirements.txt, excluding: torch family (installed above, from the CUDA index -
# letting this reinstall them from the default index would silently swap back to +cpu, same
# issue hit on Windows); triton (not needed - infer_pipeline.py never calls torch.compile, and
# it complicates version-matching for no benefit here); mmcv/mmengine/mmdet/mmpretrain (installed
# separately below via plain pip + find-links, deliberately not through `mim` - see above).
import re
lines = open('requirements.txt', encoding='utf-8').read().splitlines()
skip = re.compile(r'^\s*(torch|torchvision|torchaudio|triton|mmcv|mmengine|mmdet|mmpretrain)\s*(==?.*)?$')
kept = [l for l in lines if not skip.match(l)]
open('requirements-colab-filtered.txt', 'w', encoding='utf-8').write('\n'.join(kept) + '\n')
print(f'{len(lines)} lines -> {len(kept)} kept (excluded {len(lines) - len(kept)})')

!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q -r requirements-colab-filtered.txt

In [ ]:
# Plain pip + OpenMMLab's own wheel index, NOT `mim install` - see the markdown cell above for
# why (mim pulls in openxlab, which clobbers setuptools back to a Python-3.13-incompatible pin).
# mmengine/mmdet/mmpretrain are pure Python (no compiled extensions) - plain PyPI is fine for
# those. mmcv has compiled CUDA extensions and needs OpenMMLab's own prebuilt-wheel index,
# matched to the torch/cuda pair actually installed (torch2.1.0/cu121, per the cell above).
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q mmengine==0.10.5
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q "mmdet==3.3.0"
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q mmpretrain

!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/python -c "import mmcv, mmdet, mmengine; from mmdet.apis import init_detector; print('mmcv', mmcv.__version__, '| mmdet', mmdet.__version__, '| mmengine', mmengine.__version__)"

In [ ]:
# Gaps found while getting this running (see docs/checkpoints.md in the main project for detail),
# all installed into the Python 3.10 venv, not the notebook's own Python 3.13 kernel:
#   - opencc/zhconv: imported by infer_pipeline.py, missing from requirements.txt
#   - fairscale: needed by the repo's OWN vendored AutoHDR/mmdet/ (shadows the pip-installed
#     mmdet package when running from inside the repo dir) - models/necks/sfp.py's
#     checkpoint_wrapper. Builds from source (no prebuilt wheel), which is exactly why Python
#     3.10 matters here - Python 3.13's removed pkgutil.ImpImporter broke pkg_resources during
#     that build no matter what setuptools version was tried; Python 3.10 doesn't have that
#     problem at all (ImpImporter wasn't removed until 3.12).
#   - huggingface_hub: unpinned in requirements.txt, so pip grabs latest, which removed
#     cached_download() that diffusers==0.22.0 needs at import time. 0.25.2 satisfies both
#     diffusers==0.22.0 (needs cached_download, removed in 0.26.0) and transformers==4.45.0
#     (needs >=0.23.2) - and predates DDUFEntry (added in 0.27.0, which some other installed
#     package's import path wants - hence pinning this exactly rather than leaving it loose).
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q opencc-python-reimplemented zhconv fairscale "huggingface_hub==0.25.2"

# Belt-and-suspenders: force these two back to the known-compatible pair as the LAST install
# step, in case anything above silently upgraded either one again.
!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/pip install -q --force-reinstall --no-deps diffusers==0.22.0 "huggingface_hub==0.25.2"

!PYTHONPATH= MPLBACKEND=Agg /content/venv310/bin/python -c "\
import diffusers, huggingface_hub; \
print('diffusers', diffusers.__version__, '| huggingface_hub', huggingface_hub.__version__); \
assert diffusers.__version__ == '0.22.0'; \
assert huggingface_hub.__version__ == '0.25.2'; \
from diffusers import UNet2DModel; \
from mmdet.apis import init_detector; \
from opencc import OpenCC; \
from zhconv import convert; \
import fairscale; \
print('all imports OK')"

## 3. Extract and place checkpoints

Extracts every `*.zip` in `CKPT_SRC` into a local staging dir (`/content/ckpt_staging/` — local disk, not Drive, since unzip-over-Drive-mount is slow and sometimes flaky), then **searches recursively by basename** for each file/folder the repo actually needs and copies just that into place. This is deliberately layout-agnostic — it doesn't matter whether a zip's internal structure is flat or nested, or what the zip itself is named, since we don't fully know/control how each BaiduYun download is packaged.

What each target actually is (confirmed by inspecting the real downloads on Windows first):
- `ckpt/AutoHDR-Qwen2-1.5B/` — HF checkpoint folder, identified by containing `model.safetensors`
- `ckpt/unet/` — HF diffusers folder, identified by containing `diffusion_pytorch_model.safetensors`
- `ckpt/damage_detect.pth` — a single file, matched by exact name
- `dist/det_model/` — a folder containing the `det_model` ELF executable **and** its `_internal/` sibling (PyInstaller bundle) — copies the whole parent folder, not just the binary
- `dist/model_exe` — a single ELF executable file, found in the OCR zip under the name `reg_model` and renamed to `model_exe` (the name `utils/reg_wrapper.py` hardcodes)

In [ ]:
import glob, os, shutil, zipfile

STAGING = '/content/ckpt_staging'
shutil.rmtree(STAGING, ignore_errors=True)
os.makedirs(STAGING, exist_ok=True)

for zpath in glob.glob(os.path.join(CKPT_SRC, '*.zip')):
    name = os.path.splitext(os.path.basename(zpath))[0]
    out_dir = os.path.join(STAGING, name)
    print(f'extracting {zpath} -> {out_dir}')
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(out_dir)

# Also index any loose (non-zip) files placed directly in CKPT_SRC, e.g. a raw damage_detect.pth.
SEARCH_ROOTS = [STAGING, CKPT_SRC]

print('\nstaged contents:')
for root, dirs, files in os.walk(STAGING):
    depth = root[len(STAGING):].count(os.sep)
    if depth <= 2:
        print(' ', root)

In [ ]:
def find_dir_containing(marker_filename, roots=SEARCH_ROOTS):
    """Return the first directory (under any of `roots`) that directly contains a file
    named `marker_filename`, or None."""
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if marker_filename in filenames:
                return dirpath
    return None

def find_file(exact_filename, roots=SEARCH_ROOTS):
    """Return the first file path (under any of `roots`) with basename `exact_filename`, or None."""
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if exact_filename in filenames:
                return os.path.join(dirpath, exact_filename)
    return None

def place_dir(marker_filename, dst_path, label):
    src_dir = find_dir_containing(marker_filename, roots=SEARCH_ROOTS)
    if src_dir is None:
        print(f'MISSING: {label} (looked for a folder containing {marker_filename}) - skipping')
        return
    shutil.copytree(src_dir, dst_path, dirs_exist_ok=True)
    print(f'OK: {label}: {src_dir} -> {dst_path}')

def place_file(exact_filename, dst_path, label):
    src_file = find_file(exact_filename, roots=SEARCH_ROOTS)
    if src_file is None:
        print(f'MISSING: {label} (looked for {exact_filename}) - skipping')
        return
    os.makedirs(os.path.dirname(dst_path) or '.', exist_ok=True)
    shutil.copy2(src_file, dst_path)
    print(f'OK: {label}: {src_file} -> {dst_path}')

os.makedirs('ckpt', exist_ok=True)
os.makedirs('dist', exist_ok=True)

place_dir('model.safetensors', 'ckpt/AutoHDR-Qwen2-1.5B', 'AutoHDR-Qwen2-1.5B')
place_dir('diffusion_pytorch_model.safetensors', 'ckpt/unet', 'DiffHDR unet')
place_file('damage_detect.pth', 'ckpt/damage_detect.pth', 'Damage Localization Model')
place_dir('det_model', 'dist/det_model', 'OCR detector (det_model dir)')
place_file('reg_model', 'dist/model_exe', 'OCR recognizer (reg_model -> model_exe)')

# The OCR binaries need the execute bit, which zip/Drive transfers often drop.
for p in ['dist/det_model/det_model', 'dist/model_exe']:
    if os.path.exists(p):
        os.chmod(p, 0o755)
        print('chmod +x', p)

print('\nFinal ckpt/:', sorted(os.listdir('ckpt')) if os.path.isdir('ckpt') else None)
print('Final dist/:', sorted(os.listdir('dist')) if os.path.isdir('dist') else None)

## 4. Smoke test

Runs the official, unmodified `infer_pipeline.py` on the repo's own bundled `example.jpg`, using the 1.5B checkpoint and the Python 3.10 venv (`/content/venv310/bin/python`, built in step 2) - not the notebook's own Python 3.13 kernel. Confirmed on native Windows (before the OCR-binary blocker) that this same command gets all the way through model construction — only the checkpoint files were missing there. If step 3 above printed any `MISSING:` lines, this cell will fail at that missing checkpoint — check the placement logic or your Drive folder first.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTHONPATH'] = ''   # see step 2's markdown - Colab env vars leak into the venv otherwise
os.environ['MPLBACKEND'] = 'Agg'
!/content/venv310/bin/python infer_pipeline.py --model_name_or_path ./ckpt/AutoHDR-Qwen2-1.5B

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('results/combined/example.jpg')
plt.figure(figsize=(10, 14))
plt.imshow(img)
plt.axis('off')
plt.title('Original (top) vs restored (bottom)')
plt.show()